### Transform Constructors Data
1. Read bronze constructors table
2. Keep only the columns required for analytics (Drop URL column)
3. Standarize columns name using snake_case (constructorId -> constructor_id)
4. Rename columns to make them more meaningful (name -> constructor_name)
5. Remove duplicate records
6. Transform value of column nationality to Title Case
7. Write the transformed data to silver constructors table

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.constructors"
silver_table = f"{catalog_name}.{silver_schema}.constructors"

In [0]:
bronze_table

In [0]:
constructors_df = spark.read.table(bronze_table)

In [0]:
from pyspark.sql import functions as F

In [0]:
constructor_dropped_df = constructors_df.drop("url")

In [0]:
constructor_renamed_df = constructor_dropped_df.withColumnRenamed("constructorId", "constructor_id") \
                                             .withColumnRenamed("name","constructor_name")

In [0]:
constructor_distinct_df = constructor_renamed_df.dropDuplicates(["constructor_id"])

In [0]:
display(constructor_distinct_df)

In [0]:
constructor_final_df = (
    constructor_distinct_df
    .withColumn("nationality",F.initcap(F.col("nationality")))
)

In [0]:
display(constructor_final_df)

In [0]:
(
    constructor_final_df
    .write
    .mode("overwrite")
    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))